# Synthetic Food Waste Forecasting Dataset
Mimics the Kaggle original (`food_waste_university_canteen.csv`) by fitting statistical priors from the real data and generating a 6-month half-hourly synthetic dataset.

## Imports

In [31]:
import os
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
from scipy.stats import pearsonr, spearmanr

## Load original Kaggle data

In [32]:
original = pd.read_csv("data/original/food_waste_university_canteen.csv")
original['Date'] = pd.to_datetime(original['Date'])
original['day_of_week'] = original['Date'].dt.dayofweek

print(f"Loaded {len(original):,} rows")
print(original.head())

Loaded 2,600 rows
        Date       Meal Canteen_Section Food_Category  Waste_Weight_kg  \
0 2025-07-30     Dinner               B    Vegetables             1.50   
1 2025-06-15  Breakfast               B          Rice             3.69   
2 2025-07-29  Breakfast               A          Soup             1.54   
3 2025-07-17  Breakfast               A          Soup             1.81   
4 2025-07-03     Dinner               D          Rice             4.69   

   Unit_Price_per_kg  Cost_Loss  day_of_week  
0                3.0       4.50            2  
1                2.0       7.38            6  
2                1.5       2.31            1  
3                1.5       2.71            3  
4                2.0       9.38            3  


## Extract statistical priors from real data

In [33]:
waste_positive = original[original['Waste_Weight_kg'] > 0]
overall_mean = waste_positive['Waste_Weight_kg'].mean()
overall_std  = waste_positive['Waste_Weight_kg'].std()

meal_stats = {}
for meal in ['Breakfast', 'Lunch', 'Dinner']:
    data = waste_positive[waste_positive['Meal'] == meal]['Waste_Weight_kg']
    meal_stats[meal] = {
        'mean': data.mean() if len(data) > 5 else overall_mean,
        'std':  data.std()  if len(data) > 5 else overall_std,
    }

raw_loc = (
    waste_positive.groupby('Canteen_Section')['Waste_Weight_kg'].mean() / overall_mean
).to_dict() if 'Canteen_Section' in waste_positive.columns else {}

location_multiplier = raw_loc or {
    'Main Hall': 1.15, 'Cafe': 0.90, 'East Wing': 1.05, 'West Wing': 0.95
}

raw_dow = (
    waste_positive.groupby('day_of_week')['Waste_Weight_kg'].mean() / overall_mean
).to_dict()
dow_multiplier = raw_dow if len(raw_dow) >= 5 else {
    0: 1.05, 1: 1.10, 2: 1.08, 3: 1.05, 4: 0.95, 5: 0.75, 6: 0.65
}

print(f"Overall waste: mean={overall_mean:.2f}, std={overall_std:.2f}")
print(f"Location multipliers: {location_multiplier}")
print(f"Day-of-week multipliers: {dow_multiplier}")

Overall waste: mean=2.59, std=1.41
Location multipliers: {'A': 0.9720387461047092, 'B': 1.0108876237334945, 'C': 1.0224485866827961, 'D': 0.9947661182421882}
Day-of-week multipliers: {0: 0.9742805501037202, 1: 0.9442401441321133, 2: 0.9857975979535906, 3: 1.0425593739981225, 4: 0.9755101123831222, 5: 1.0207426237487853, 6: 1.046505056093712}


## Build continuous half-hourly timeline

In [34]:
RNG   = np.random.default_rng(seed=2025)
START = "2025-01-01 00:00"
END   = "2025-06-30 23:30"

idx = pd.date_range(start=START, end=END, freq="30min")
n   = len(idx)
print(f"Timeline: {idx[0]} → {idx[-1]}  ({n:,} rows, {n/48:.0f} days)")

df = pd.DataFrame({"timestamp": idx})
df["date"]        = df["timestamp"].dt.date
df["year"]        = df["timestamp"].dt.year
df["month"]       = df["timestamp"].dt.month
df["day"]         = df["timestamp"].dt.day
df["hour"]        = df["timestamp"].dt.hour
df["minute"]      = df["timestamp"].dt.minute
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["week_of_year"]= df["timestamp"].dt.isocalendar().week.astype(int)
df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)
df["hour_frac"]   = df["hour"] + df["minute"] / 60.0
df["day_index"]   = (df["timestamp"] - df["timestamp"].iloc[0]).dt.days

Timeline: 2025-01-01 00:00:00 → 2025-06-30 23:30:00  (8,688 rows, 181 days)


## Add cyclical time features

In [35]:
df["hour_sin"]  = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"]  = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

print("Cyclical features added.")

Cyclical features added.


## Assign meal type and location

In [36]:
def assign_meal(hour):
    if   6  <= hour < 10: return 'Breakfast'
    elif 11 <= hour < 14: return 'Lunch'
    elif 17 <= hour < 21: return 'Dinner'
    else:                 return 'Closed'

df['meal'] = df['hour'].apply(assign_meal)

locations = list(location_multiplier.keys())
df['location_id'] = RNG.choice(locations, size=n)

open_mask   = (df['meal'] != 'Closed').values
closed_mask = ~open_mask

print(df['meal'].value_counts())

meal
Closed       4706
Breakfast    1448
Dinner       1448
Lunch        1086
Name: count, dtype: int64


## Generate footfall FIRST

Footfall is the cause and waste is the effect.

Gaussian peaks are parameterised separately for weekday vs weekend:
university attendance is lower on weekends.

In [37]:
MEAL_FOOTFALL = {
    'Breakfast': {'center': 7.5,  'sigma': 0.9,  'weekday': 85,  'weekend': 45},
    'Lunch':     {'center': 12.5, 'sigma': 1.0,  'weekday': 175, 'weekend': 100},
    'Dinner':    {'center': 18.5, 'sigma': 1.2,  'weekday': 130, 'weekend': 85},
}

hour_frac_arr  = df['hour_frac'].values
is_weekend_arr = df['is_weekend'].values

base_footfall = np.zeros(n, dtype=float)
for meal, p in MEAL_FOOTFALL.items():
    peak = np.where(is_weekend_arr, p['weekend'], p['weekday']).astype(float)
    base_footfall += peak * np.exp(
        -0.5 * ((hour_frac_arr - p['center']) / p['sigma']) ** 2
    )

unique_dates   = sorted(df['date'].unique())
n_days         = len(unique_dates)
day_effects    = RNG.lognormal(mean=0.0, sigma=0.18, size=n_days)  # E≈1, CV≈18%
date_to_effect = dict(zip(unique_dates, day_effects))
day_effect_arr = df['date'].map(date_to_effect).values

trend = 1.0 + 0.0008 * df['day_index'].values

dow_factor = df['day_of_week'].map(dow_multiplier).fillna(1.0).values
loc_factor = df['location_id'].map(location_multiplier).fillna(1.0).values

footfall_mean = base_footfall * dow_factor * loc_factor * day_effect_arr * trend
footfall_mean = np.clip(footfall_mean, 0.0, 300.0)

footfall = np.zeros(n, dtype=int)
footfall[open_mask]   = RNG.poisson(np.clip(footfall_mean[open_mask], 0, 300))
footfall[closed_mask] = RNG.integers(0, 6, size=closed_mask.sum())
footfall = np.clip(footfall, 0, 300).astype(int)

df['footfall_raw'] = footfall.copy()
df['footfall']     = footfall.copy()

print(f"footfall (open): mean={footfall[open_mask].mean():.1f}, "
      f"std={footfall[open_mask].std():.1f}, max={footfall[open_mask].max()}")

footfall (open): mean=79.3, std=50.6, max=293


## Apply holiday flags and multipliers — to footfall first

Unlike public venues a university canteen empties when students travel home for breaks. 
Multipliers reflect actual campus attendance patterns
- breaks (Easter, Memorial Day, New Year)
- Social/cultural events on campus

In [38]:
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=START, end=END)
extra_holidays = pd.to_datetime([
    "2025-01-31", "2025-02-14", "2025-02-17", "2025-03-17",
    "2025-04-18", "2025-04-20", "2025-05-05", "2025-06-15", "2025-06-19"
])
all_holidays  = holidays.union(extra_holidays)
holiday_dates = set(all_holidays.date)
df["is_holiday"] = df["date"].apply(lambda d: int(d in holiday_dates))

holiday_name_map = {
    pd.Timestamp("2025-01-01").date(): "New Year's Day",
    pd.Timestamp("2025-01-20").date(): "MLK Day",
    pd.Timestamp("2025-02-14").date(): "Valentine's Day",
    pd.Timestamp("2025-02-17").date(): "Presidents Day",
    pd.Timestamp("2025-03-17").date(): "St. Patrick's Day",
    pd.Timestamp("2025-04-18").date(): "Good Friday",
    pd.Timestamp("2025-04-20").date(): "Easter",
    pd.Timestamp("2025-05-05").date(): "Cinco de Mayo",
    pd.Timestamp("2025-05-26").date(): "Memorial Day",
    pd.Timestamp("2025-06-15").date(): "Father's Day",
    pd.Timestamp("2025-06-19").date(): "Juneteenth",
}
for hd in all_holidays:
    if hd.date() not in holiday_name_map:
        holiday_name_map[hd.date()] = "Holiday"
df["holiday_name"] = df["date"].map(holiday_name_map).fillna("")

holiday_mult_map = {
    "New Year's Day":    0.15,   # campus virtually empty
    "MLK Day":           0.55,   # partial attendance, some students remain
    "Valentine's Day":   1.20,   # mild campus social bump (on-campus students)
    "Presidents Day":    0.55,   # many students travel home
    "St. Patrick's Day": 1.10,   # mild cultural event on campus
    "Good Friday":       0.50,   # students begin Easter travel
    "Easter":            0.30,   # campus largely empty
    "Cinco de Mayo":     1.10,   # mild cultural event
    "Memorial Day":      0.35,   # semester near end, campus emptying
    "Father's Day":      0.65,   # some students stay
    "Juneteenth":        0.80,   # mild reduction
    "Holiday":           0.80,   # generic: mild reduction
}
df['holiday_mult'] = df['holiday_name'].map(lambda nm: holiday_mult_map.get(nm, 1.0))
df.loc[open_mask, 'footfall'] = (
    df.loc[open_mask, 'footfall'].astype(float) * df.loc[open_mask, 'holiday_mult']
).round().astype(int)
df['footfall_raw'] = df['footfall'].copy()

print(f"Holidays flagged: {df['is_holiday'].sum():,} rows")
print(f"Holiday footfall (open):    {df.loc[open_mask & (df['is_holiday'].values==1), 'footfall'].mean():.1f} avg")
print(f"Non-holiday footfall (open): {df.loc[open_mask & (df['is_holiday'].values==0), 'footfall'].mean():.1f} avg")

Holidays flagged: 576 rows
Holiday footfall (open):    53.3 avg
Non-holiday footfall (open): 79.1 avg


## Apply special event boosts to footfall

In [39]:
special_events = {
    "2025-01-12": "NFL Playoff Watch Party",
    "2025-02-02": "Super Bowl Watch Party",
    "2025-02-08": "Valentine's Dinner Promo",
    "2025-03-22": "March Madness Weekend",
    "2025-03-23": "March Madness Weekend",
    "2025-04-04": "Campus Career Fair",
    "2025-04-05": "Campus Career Fair",
    "2025-04-27": "Spring Festival",
    "2025-05-10": "Mother's Day Brunch",
    "2025-05-24": "Memorial Weekend BBQ",
    "2025-05-25": "Memorial Weekend BBQ",
    "2025-06-07": "Outdoor Concert Night",
    "2025-06-21": "Summer Solstice Bash",
    "2025-06-28": "Pride Month Event",
}
event_date_map  = {pd.Timestamp(k).date(): v for k, v in special_events.items()}
df["special_event"] = df["date"].map(event_date_map).fillna("")

event_open_mask = (df["special_event"] != "").values & open_mask

df.loc[event_open_mask, 'footfall'] = (
    df.loc[event_open_mask, 'footfall'].astype(float)
    * RNG.uniform(2.2, 3.0, event_open_mask.sum())
).round().astype(int)

df['footfall'] = np.clip(df['footfall'], 0, 500).astype(int)

print(f"Special event open rows: {event_open_mask.sum():,}")
print(f"Event footfall avg:  {df.loc[event_open_mask, 'footfall'].mean():.1f}")
print(f"Normal footfall avg: {df.loc[open_mask & ~event_open_mask & (df['is_holiday'].values==0), 'footfall'].mean():.1f}")

Special event open rows: 308
Event footfall avg:  142.8
Normal footfall avg: 81.3


## Generate waste_kg FROM footfall

In [40]:
WASTE_PARAMS = {
    'Breakfast': {'overhead': 0.60, 'per_person': 0.010},
    'Lunch':     {'overhead': 1.00, 'per_person': 0.015},
    'Dinner':    {'overhead': 0.70, 'per_person': 0.012},
    'Closed':    {'overhead': 0.00, 'per_person': 0.000},
}

overhead_arr   = df['meal'].map({m: p['overhead']   for m, p in WASTE_PARAMS.items()}).fillna(0).values
per_person_arr = df['meal'].map({m: p['per_person'] for m, p in WASTE_PARAMS.items()}).fillna(0).values
waste_mean = (overhead_arr + per_person_arr * df['footfall'].values) * loc_factor * trend

CV          = 0.15
alpha_shape = (1.0 / CV) ** 2        # ≈ 44.44
unit_gamma  = RNG.gamma(alpha_shape, 1.0, size=n)
waste_kg    = np.where(
    waste_mean > 0,
    unit_gamma * (waste_mean / alpha_shape),
    0.0
)

waste_kg[closed_mask] = RNG.uniform(0.0, 0.05, closed_mask.sum())

waste_kg = np.clip(np.round(waste_kg, 2), 0.0, 12.0)
waste_kg[closed_mask] = np.round(np.clip(waste_kg[closed_mask], 0.0, 0.05), 2)

df['waste_kg'] = waste_kg

print(f"waste_kg overall: mean={df['waste_kg'].mean():.3f}, std={df['waste_kg'].std():.3f}")
print(f"Open hours:       mean={df.loc[open_mask, 'waste_kg'].mean():.3f}")
print(f"Closed hours:     mean={df.loc[closed_mask, 'waste_kg'].mean():.4f}")
print(f"\nBy meal:")
print(df.groupby('meal')['waste_kg'].agg(['mean','std','min','max']).round(3))

waste_kg overall: mean=0.912, std=1.221
Open hours:       mean=1.960
Closed hours:     mean=0.0249

By meal:
            mean    std   min   max
meal                               
Breakfast  1.133  0.397  0.49  3.13
Closed     0.025  0.015  0.00  0.05
Dinner     1.919  0.722  0.68  6.05
Lunch      3.116  1.155  0.90  9.42


## Waste category breakdown (organic / recyclable / landfill)

In [41]:
org_frac  = 0.52 + 0.12 * np.sin(2 * np.pi * (df["hour_frac"] - 12) / 24)
rec_frac  = 0.28 - 0.06 * np.sin(2 * np.pi * (df["hour_frac"] - 18) / 24)
land_frac = 1 - org_frac - rec_frac

w = df["waste_kg"].fillna(0).values
df["waste_organic_kg"]    = np.round(np.clip(w * org_frac,  0, None), 2)
df["waste_recyclable_kg"] = np.round(np.clip(w * rec_frac,  0, None), 2)
df["waste_landfill_kg"]   = np.round(np.clip(w * land_frac, 0, None), 2)

print("Waste categories created.")
print(df[["waste_organic_kg", "waste_recyclable_kg", "waste_landfill_kg"]].describe().round(3))

Waste categories created.
       waste_organic_kg  waste_recyclable_kg  waste_landfill_kg
count          8688.000             8688.000           8688.000
mean              0.495                0.279              0.137
std               0.687                0.393              0.174
min               0.000                0.000              0.000
25%               0.010                0.010              0.000
50%               0.020                0.010              0.010
75%               0.850                0.450              0.250
max               5.050                3.200              1.180


## Build target variable and select final columns

In [42]:
df["waste_kg_target"] = df["waste_kg"].clip(lower=0)
df["footfall"]        = df["footfall"].astype(int)

final_cols = [
    "timestamp", "location_id", "meal",
    "year", "month", "day", "hour", "minute",
    "day_of_week", "week_of_year", "is_weekend",
    "hour_frac", "hour_sin", "hour_cos", "month_sin", "month_cos",
    "is_holiday", "holiday_name", "special_event",
    "footfall", "footfall_raw",
    "waste_kg", "waste_kg_target",
    "waste_organic_kg", "waste_recyclable_kg", "waste_landfill_kg",
]

df_final = df[final_cols].sort_values("timestamp").reset_index(drop=True)
print(f"Final shape: {df_final.shape}")
print(df_final.dtypes)

Final shape: (8688, 26)
timestamp              datetime64[us]
location_id                       str
meal                              str
year                            int32
month                           int32
day                             int32
hour                            int32
minute                          int32
day_of_week                     int32
week_of_year                    int64
is_weekend                      int64
hour_frac                     float64
hour_sin                      float64
hour_cos                      float64
month_sin                     float64
month_cos                     float64
is_holiday                      int64
holiday_name                      str
special_event                     str
footfall                        int64
footfall_raw                    int64
waste_kg                      float64
waste_kg_target               float64
waste_organic_kg              float64
waste_recyclable_kg           float64
waste_landfill_kg         

## Save

In [43]:
OUT = "data/synthetic_forecast_ready.csv"
df_final.to_csv(OUT, index=False)
print(f"Saved {OUT}  ({len(df_final):,} rows × {len(df_final.columns)} cols)")

Saved data/synthetic_forecast_ready.csv  (8,688 rows × 26 cols)


## validation and checks

In [44]:
open_df = df_final[df_final['meal'] != 'Closed'].copy()

r_p, p_p = pearsonr(open_df['footfall'], open_df['waste_kg'])
r_s, p_s = spearmanr(open_df['footfall'], open_df['waste_kg'])
print(f"Q1  footfall - waste  Pearson r={r_p:.3f}  Spearman r={r_s:.3f}")
assert r_p > 0.70, f"Signal too weak: Pearson r={r_p:.3f}. Check causal params."

meal_means = open_df.groupby('meal')['waste_kg'].mean()
print(f"\nQ2  Waste by meal:")
print(meal_means.round(3))
assert meal_means['Lunch'] > meal_means['Breakfast'], "Lunch must exceed Breakfast"
assert df_final[df_final['meal']=='Closed']['waste_kg'].mean() < 0.10, \
    "Closed hours must be near zero"

is_event  = df_final['special_event'] != ''
is_open   = df_final['meal'] != 'Closed'
is_normal = ~is_event & (df_final['is_holiday'] == 0) & is_open
normal_mean = df_final.loc[is_normal, 'waste_kg'].mean()
event_mean  = df_final.loc[is_event & is_open, 'waste_kg'].mean()
print(f"\nQ3  Event lift: {event_mean:.3f} vs normal {normal_mean:.3f}  → {event_mean/normal_mean:.2f}×")
assert event_mean > normal_mean * 1.20, "Event days must show ≥ 20% lift"

holiday_mean = df_final.loc[(df_final['is_holiday']==1) & is_open, 'waste_kg'].mean()
print(f"Q4  Holiday reduction: {holiday_mean:.3f} vs normal {normal_mean:.3f}  → {holiday_mean/normal_mean:.2f}×")
assert holiday_mean < normal_mean * 0.90, "Holiday days must show ≥ 10% reduction"

dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_summary = open_df.groupby('day_of_week')['waste_kg'].mean().round(3)
dow_summary.index = [dow_labels[i] for i in dow_summary.index]
print(f"\nQ5  Waste by day of week:")
print(dow_summary)

print(f"\nQ6  footfall std={df_final['footfall'].std():.1f}  "
      f"range=[{df_final['footfall'].min()}, {df_final['footfall'].max()}]")
assert df_final['footfall'].std() > 20, "footfall lacks variability"

n_days_covered = df_final['timestamp'].dt.date.nunique()
print(f"Q7  {n_days_covered} unique days  (min recommended: 90)")
assert n_days_covered >= 90

orig_waste  = original['Waste_Weight_kg'].dropna()
synth_waste = df_final.loc[is_open, 'waste_kg']
print(f"\nQ8  Original (open-hour equiv):  mean={orig_waste.mean():.3f}  std={orig_waste.std():.3f}")
print(f"    Synthetic (open hours):       mean={synth_waste.mean():.3f}  std={synth_waste.std():.3f}")

print("check are passed dataset is ready")

Q1  footfall - waste  Pearson r=0.908  Spearman r=0.912

Q2  Waste by meal:
meal
Breakfast    1.133
Dinner       1.919
Lunch        3.116
Name: waste_kg, dtype: float64

Q3  Event lift: 2.814 vs normal 1.916  → 1.47×
Q4  Holiday reduction: 1.525 vs normal 1.916  → 0.80×

Q5  Waste by day of week:
Mon    1.874
Tue    1.937
Wed    1.978
Thu    2.148
Fri    2.061
Sat    1.959
Sun    1.761
Name: waste_kg, dtype: float64

Q6  footfall std=56.4  range=[0, 445]
Q7  181 unique days  (min recommended: 90)

Q8  Original (open-hour equiv):  mean=2.586  std=1.408
    Synthetic (open hours):       mean=1.960  std=1.106
check are passed dataset is ready
